## Шаг 1. Настройка Spark-агрегации

In [ ]:
#filename=sample_file.py

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Создаем Spark-сессию
spark = SparkSession.builder.appName('payment_type_agg') \
.config("sample_cnf", "sample_cnf").getOrCreate()

# Указываем параметры для Clickhouse
jdbcPort = 8443
jdbcHostname = 'sample.net'
jdbcDatabase = 'sample_db'
jdbcURL = f'jdbc:clickhouse://{jdbcHostname}:{jdbcPort}/{jdbcDatabase}?ssl=true'

# Считываем данные, создаем датафрейм
df = spark.read.parquet('s3a://sample_bucket/sample_path.parquet')

# Создаем агрегат по способам оплаты
grouped_df = df.groupBy('payment_type').agg(
    F.count('*').alias('runs_count'),
    F.avg('fare').alias('avg_fare'),
    F.avg('tips').alias('avg_tips'),
    F.sum('trip_total').alias('sum_trip_total')
)

grouped_df.write.format('jdbc') \
.option('url', jdbcURL) \
.option('user', 'sample_user') \
.option('password', 'sample_password') \
.option('dbtable', 'taxi_payment_summary') \
.option("createTableOptions", "ENGINE=MergeTree() ORDER BY payment_type") \
.mode('overwrite') \
.save()

## Шаг 2. Настройте DAG

In [ ]:
from datetime import datetime
from airflow import DAG
from airflow.sensors.s3_key_sensor import S3KeySensor
from airflow.providers.yandex.operators.dataproc import DataprocCreatePysparkJobOperator

user = 'sample_user'
DAG_ID = "payment_type_aggregation"

with DAG (
    DAG_ID,
    start_date = datetime(2026, 6, 8),
    schedule = '@daily',
    catchup = False
) as dag:
    # Ждем появления файла в S3
    wait_for_file = S3KeySensor(
    task_id = 'wait_for_taxi_data_file',
    aws_conn_id = 's3',
    poke_interval = 60 * 5,
    timeout = 60 * 60,
    bucket_name = 'sample_bucket',
    bucket_key = 'sample_path.parquet',
    mode = 'poke',
    wildcard_match = False
    )
    # Запускаем PySpark
    run_pyspark = DataprocCreatePysparkJobOperator(
    name = 'create_payment_type_agg_load_to_ch',
    task_id = 'launch_spark_code',
    cluster_id = 'sample_cluster',
    main_python_file_uri = f's3a://sample_bucket/{user}/sample_path.py'
    )
    
    # Зависимости
    wait_for_file >> run_pyspark